In [0]:
%run ./00_setup

In [0]:
import great_expectations as gx

orders_df = spark.table(f"workspace.bronze.orders")
customers_keys = (
    spark.table(f"workspace.bronze.customers")
    .select(F.col("customer_id").alias("customer_id_ref"))
    .distinct()
)

orders_fk_df = (
    orders_df
    .join(customers_keys, orders_df.customer_id == customers_keys.customer_id_ref, "left")
    .withColumn("customer_exists", F.col("customer_id_ref").isNotNull())

)

In [0]:
orders_fk_df.display()

In [0]:
from pyspark.sql import functions as F

# Filtramos donde el valor es False
df_errors = orders_fk_df.filter(F.col("customer_exists") == False)

# Mostramos el resultado
display(df_errors)

In [0]:

context = gx.get_context()

suite_name = "orders_fk_suite"
context.add_or_update_expectation_suite(expectation_suite_name=suite_name)

# 1. Tu configuración original 
datasource_config = {
    "name": "spark_orders_ds",
    "class_name": "Datasource",
    "execution_engine": {
        "class_name": "SparkDFExecutionEngine",
        "persist": False #
    },
    "data_connectors": {
        "default_runtime_data_connector_name": {
            "class_name": "RuntimeDataConnector",
            "batch_identifiers": ["default_identifier_name"]
        }
    }
}

# 2. APLICAR la configuración al contexto (esto faltaba)
context.add_datasource(**datasource_config)

# 3. Crear el Batch Request compatible con la configuración de arriba
batch_request = gx.core.batch.RuntimeBatchRequest(
    datasource_name="spark_orders_ds",
    data_connector_name="default_runtime_data_connector_name",
    data_asset_name="orders_fk_asset",
    runtime_parameters={"batch_data": orders_fk_df},
    batch_identifiers={"default_identifier_name": "default_identifier"}
)

# 4. Obtener el validador pasando el Batch Request
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=suite_name
)

# 5. Ejecutar tu expectativa de llave foránea (Foreign Key)
validator.expect_column_values_to_be_in_set(
    "customer_exists", 
    [True]
)

# Guardar y validar
validator.save_expectation_suite(discard_failed_expectations=False)
fk_result = validator.validate()

display(fk_result.to_json_dict())

save_json(
    f"{VALIDATION_PATH}/orders_fk_validation.json",
    fk_result.to_json_dict()
)

In [0]:
customers_validation = load_json(f"{VALIDATION_PATH}/customers_validation.json")

orders_validation = load_json(f"{VALIDATION_PATH}/orders_validation.json")

fk_validation = load_json(f"{VALIDATION_PATH}/orders_fk_validation.json")


In [0]:
customers_validation

In [0]:
orders_validation

In [0]:
fk_validation

In [0]:
gate_summary = [
    ("customers_suite", customers_validation["success"]),
    ("orders_suite", orders_validation["success"]),
    ("orders_fk_suite", fk_validation["success"]),
]

gate_df = spark.createDataFrame(gate_summary, ["suite_name", "success"])
display(gate_df)

overall_gate = all([r[1] for r in gate_summary])
print("QUALITY GATE =", "PASS" if overall_gate else "FAIL")